<a href="https://colab.research.google.com/github/tang1114/Thai-SMS-Spam-Detection/blob/main/Thai_SMS_Spam_Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Tang**

In [ ]:
!pip install pandas scikit-learn pythainlp emoji numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 38.0 MB/s eta 0:00:00


In [ ]:
# --- ส่วนที่ 2: Feature Engineering + TF-IDF + Export (spam version) ---

import pandas as pd
import re
import string
import emoji
import nltk
from pythainlp.tokenize import word_tokenize
from pythainlp.corpus.common import thai_stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from nltk.corpus import stopwords
import requests

try:
    stopwords.words('english')
except LookupError:
    nltk.download('stopwords')

# --- Load Data ---
def load_data_from_url(url, label):
    try:
        response = requests.get(url)
        # Check if the request was successful
        response.raise_for_status()
        lines = response.text.splitlines()
        return [line.strip().split('|')[1] for line in lines if line.startswith(label + '|')]
    except requests.exceptions.RequestException as e:
        print(f"Error fetching data from {url}: {e}")
        return []

# Raw URLs for the files on GitHub
spam_url = 'https://raw.githubusercontent.com/0-Mini-Peak-1/Thai-SMS-Scam-Spam-OTP-Dataset/refs/heads/main/spam.txt'
ok_url = 'https://raw.githubusercontent.com/0-Mini-Peak-1/Thai-SMS-Scam-Spam-OTP-Dataset/refs/heads/main/ok.txt'

# Load the data using the new function
spam_messages = load_data_from_url(spam_url, 'spam')
ok_messages = load_data_from_url(ok_url, 'OK') # <-- Added this line

# Combine data and create labels
texts = spam_messages + ok_messages
labels = [1] * len(spam_messages) + [0] * len(ok_messages)

# --- Stopwords and spam word setup ---
thai_sw = set(thai_stopwords())
english_sw = set(stopwords.words('english'))
custom_sw = { 'https', 'http', 'วันที่', 'ขอบคุณ', 'ท่าน', 'www', 'com', 'bit',
    'ค่ะ', 'ครับ', 'คือ', 'ว่า', 'ก็', 'และ', 'หรือ', 'กับ', 'ให้', 'ได้',
    'ไม่', 'มี', 'แล้ว', 'ยัง', 'จะ', 'อย่าง', 'เพียง', 'แต่', 'เมื่อ',
    'จาก', 'ที่', 'ใน', 'ของ', 'โดย', 'บน', 'ซึ่ง', 'ทั้ง', 'อีก', 'จึง',
    'เพราะ', 'เลย', 'ก็ยัง', 'ก็ได้', 'จ้า', 'นะ', 'ล่ะ', 'เดี๋ยว', 'แค่',
    'มาก', 'น้อย', 'เป็น'}
all_stopwords = thai_sw.union(custom_sw).union(english_sw).union(set(string.punctuation))

custom_spamwords= {'ฟรี', 'แถม', 'พิเศษ', '100%', 'ด่วน', 'อันดับ1', 'ถอนไว', 'โบนัส', 'แตกง่าย', 'แจก', 'ลุ้น'}
promo_words = {'สมัคร', 'ฟรี', 'แถม', 'พิเศษ', 'โบนัส', 'แจก', 'ลด', 'สิทธิ์', 'โปร', 'ดีล', 'คูปอง', 'ส่วนลด', 'สมัครด่วน', 'ยืนยันสิทธิ'}
money_words = {'บาท', 'เงิน', 'จ่าย', 'ค่า', 'บ.', 'บัตรเติมเงิน', 'ชำระ', 'จ่ายเงิน', 'รับเงิน', 'คืนเงิน', 'มูลค่า'}
number_words = set([str(i) for i in range(0, 101)]) | {'เลข', 'เลขเด็ด', 'งวด', 'หวย', 'รางวัล', 'เลขท้าย'}
phone_words = {'โทร', 'ทรู', 'เน็ต', 'เติม', 'ไลน์', 'เบอร์', 'มือถือ', 'โทรศัพท์', 'ติดต่อ', 'line', 'เบอร์โทร'}
urgent_words = {'ด่วน', 'คลิก', 'รีบ', 'อย่า', 'สมัครด่วน', 'ยืนยันสิทธิ', 'อย่ารอช้า', 'เฉพาะคุณ', 'กด', 'เร็ว', 'ทันที'}
luck_words = {'ดวง', 'ดวงดี', 'เฮง', 'โชค', 'โชคดี', 'รวย', 'มงคล', 'หวย', 'รางวัล'}
product_words = {'สินค้า', 'บริการ', 'แพ็ก', 'แพ็กเกจ', 'ดีล', 'ช้อป', 'โปร', 'ส่วนลด', 'คูปอง', 'โปรโมชัน', 'โปรโมชั่น'}

# --- Preprocessing ---
def preprocess(text):
    emoji_count = count_emoji(text)
    translator = str.maketrans('', '', string.punctuation)
    text_cleaned = text.translate(translator)
    tokens = word_tokenize(text_cleaned, engine='newmm', keep_whitespace=False)
    tokens = [t for t in tokens if t.lower() not in all_stopwords and len(t) > 2 and not t.isdigit()]


    return tokens, emoji_count

# --- Feature functions ---
def count_urls(text):
    return len(re.findall(r'(http[s]?://|www\.|bit\.ly|cutt\.ly|\.com|\.me|\.co|\.in)', text))

def count_money(text):
    return len(re.findall(r'\d+[ ]*(บาท|บ|฿|บ\.)', text))

def count_digits(text):
    return len(re.findall(r'\d+', text))

def has_line_id(text):
    return int('line' in text.lower() or 'ไลน์' in text.lower())

def has_phone_number(text):
    return int(bool(re.search(r'\b0[689]{1}[0-9]{8}\b', text)))

def spamword_count(tokens):
    return sum(1 for w in tokens if w in custom_spamwords)

def message_length(text):
    return len(text)

def word_count(tokens):
    return len(tokens)

def digit_ratio(text):
    digits = sum(c.isdigit() for c in text)
    return digits / len(text) if len(text) > 0 else 0

def count_emoji(text):
    return sum(1 for ch in text if emoji.is_emoji(ch))

def count_uppercase_en(text):
    return sum(1 for w in re.findall(r'[A-Z]{2,}', text))

def count_short_link(text):
    return len(re.findall(r'(bit\.ly|tmn\.app\.link|ct\.elinks\.io|m9s\.in|shp\.ee|store\.line\.me)', text))

def count_promo_words(tokens):
    return sum(1 for w in tokens if w in promo_words)

def count_money_words(tokens):
    return sum(1 for w in tokens if w in money_words)

def count_number_words(tokens):
    return sum(1 for w in tokens if w in number_words)

def count_phone_words(tokens):
    return sum(1 for w in tokens if w in phone_words)

def count_urgent_words(tokens):
    return sum(1 for w in tokens if w in urgent_words)

def count_luck_words(tokens):
    return sum(1 for w in tokens if w in luck_words)

def count_product_words(tokens):
    return sum(1 for w in tokens if w in product_words)

def count_ussd_code(text):
    return 5 if re.search(r'\*(\d+\*)*\d+#', text) else 0

# --- Extract Features ---
all_features_list = []
spam_tokenized = [preprocess(t) for t in spam_messages]
ok_tokenized = [preprocess(t) for t in ok_messages]

for i, text in enumerate(spam_messages):
    tokens, emoji_count = spam_tokenized[i]
    all_features_list.append({
        'text': text, 'label': 1,
        'url_count': count_urls(text),
        'short_link': count_short_link(text),
        'money_count': count_money(text),
        'digit_count': count_digits(text),
        'digit_ratio': digit_ratio(text),
        'ussd_code': count_ussd_code(text),
        'spamword_count': spamword_count(tokens),
        'has_line_id': has_line_id(text),
        'has_phone_number': has_phone_number(text),
        'emoji_count': emoji_count,
        'uppercase_en_count': count_uppercase_en(text),
        'promo_word_count': count_promo_words(tokens),
        'money_word_count': count_money_words(tokens),
        'number_word_count': count_number_words(tokens),
        'phone_word_count': count_phone_words(tokens),
        'urgent_word_count': count_urgent_words(tokens),
        'luck_word_count': count_luck_words(tokens),
        'product_word_count': count_product_words(tokens),
        'msg_length': message_length(text),
        'word_count': word_count(tokens)
    })

for i, text in enumerate(ok_messages):
    tokens, emoji_count = ok_tokenized[i]
    all_features_list.append({
        'text': text, 'label': 0,
        'url_count': count_urls(text),
        'short_link': count_short_link(text),
        'money_count': count_money(text),
        'digit_count': count_digits(text),
        'digit_ratio': digit_ratio(text),
        'ussd_code': count_ussd_code(text),
        'spamword_count': spamword_count(tokens),
        'has_line_id': has_line_id(text),
        'has_phone_number': has_phone_number(text),
        'emoji_count': emoji_count,
        'uppercase_en_count': count_uppercase_en(text),
        'promo_word_count': count_promo_words(tokens),
        'money_word_count': count_money_words(tokens),
        'number_word_count': count_number_words(tokens),
        'phone_word_count': count_phone_words(tokens),
        'urgent_word_count': count_urgent_words(tokens),
        'luck_word_count': count_luck_words(tokens),
        'product_word_count': count_product_words(tokens),
        'msg_length': message_length(text),
        'word_count': word_count(tokens)
    })

feature_df = pd.DataFrame(all_features_list)

# --- TF-IDF ---
# Get the preprocessed tokens for TF-IDF calculation
all_tokens_for_tfidf = [preprocess(t)[0] for t in texts]

# Join tokens back into strings for TF-IDF Vectorizer
processed_texts_for_tfidf = [' '.join(tokens) for tokens in all_tokens_for_tfidf]

tfidf = TfidfVectorizer(
    stop_words=None,
    max_df=0.85,
    min_df=3
)
# Fit TF-IDF on the joined tokens (which now have stop words removed)
tfidf.fit(processed_texts_for_tfidf)

tfidf_matrix = tfidf.transform(processed_texts_for_tfidf).toarray()
tfidf_df = pd.DataFrame(tfidf_matrix, columns=tfidf.get_feature_names_out())


# --- Combine ---
combined_df = pd.concat([feature_df.reset_index(drop=True), tfidf_df.reset_index(drop=True)], axis=1)
combined_df.to_csv('spam_features.csv', index=False)
print("'spam_features.csv' created successfully.")

'spam_features.csv' created successfully.


In [ ]:
import pandas as pd
import numpy as np
import re
import joblib # For saving and loading the model and tools
from collections import Counter

# Scikit-learn imports
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from sklearn.feature_extraction.text import TfidfVectorizer # Import TfidfVectorizer

# TensorFlow and Keras for Neural Network
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

# PyThaiNLP for text processing in the final pipeline
from pythainlp.tokenize import word_tokenize
from pythainlp.corpus.common import thai_stopwords
import string
import nltk
import emoji # Import emoji for count_emoji function
from nltk.corpus import stopwords # Import stopwords from nltk

# Ensure NLTK data is available
try:
    nltk.data.find('corpora/stopwords')
except LookupError:
    nltk.download('stopwords')

print("Libraries imported successfully!")

# Load the dataset
try:
    df = pd.read_csv('spam_features.csv')
    print("spam_features.csv loaded successfully.")
except FileNotFoundError:
    print("Error: spam_features.csv not found.")
    print("Please make sure the CSV file is in the same directory.")
    # As a fallback, create a dummy dataframe for demonstration
    # Note: This dummy dataframe might not perfectly match the column names from cell 3
    # but will allow the code to run for demonstration purposes if the file is missing.
    df = pd.DataFrame({
        'text': ['ยินดีด้วย! คุณได้รับเครดิตฟรี 500 บาท เพียงแอดไลน์ @123xyz แล้วยืนยันตัวตนด่วน! คลิกเลย https://bit.ly/totally-a-scam',
                 'สวัสดีครับคุณสมชาย พรุ่งนี้ประชุมตอน 10 โมงเช้านะครับ',
                 'ลุ้นรับรางวัลใหญ่ แจ็คพอตแตกง่าย ถอนไว ไม่มีขั้นต่ำ สมัครเลยที่ www.scamsite.co',
                 'คุณมีพัสดุตกค้างจากไปรษณีย์ไทย กรุณาติดต่อเจ้าหน้าที่เพื่อยืนยันข้อมูลส่วนตัวที่เบอร์ 081-111-1111'],
        'label': [1, 0, 1, 1],
        'url_count': [1, 0, 1, 0],
        'short_link': [1, 0, 0, 0], # Added short_link based on user's list
        'money_count': [1, 0, 0, 0],
        'digit_count': [3, 2, 0, 10],
        'digit_ratio': [0.114754, 0.0, 0.0, 0.0], # Added digit_ratio
        'ussd_code': [0, 0, 0, 0],
        'spamword_count': [2, 0, 3, 0],
        'has_line_id': [1, 0, 0, 0],
        'has_phone_number': [0, 0, 0, 1],
        'emoji_count': [1, 0, 0, 0], # Added emoji_count
        'uppercase_en_count': [0, 0, 0, 0], # Added uppercase_en_count
        'urgent_phrase_count': [2, 0, 1, 1],
        'promo_word_count': [1, 0, 1, 0], # Added based on user's list
        'money_word_count': [1, 0, 0, 0], # Added based on user's list
        'number_word_count': [0, 0, 0, 1], # Added based on user's list
        'phone_word_count': [0, 0, 0, 1], # Added based on user's list
        'urgent_word_count': [1, 0, 1, 0], # Added based on user's list
        'luck_word_count': [1, 0, 1, 0], # Added based on user's list
        'product_word_count': [0, 0, 0, 0], # Added based on user's list
        'msg_length': [61, 40, 65, 70], # Added msg_length
        'word_count': [12, 6, 8, 11], # Added word_count
    })
    print("Created a dummy DataFrame to proceed.")


# Separate features (X) and target (y)
# Ensure 'text' and 'label' are excluded from features
X = df.drop(columns=['label', 'text'])
y = df['label']

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")

# Define models and split ratios
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Gaussian Naive Bayes': GaussianNB(),
    'Random Forest': RandomForestClassifier(random_state=42),
    'Neural Network': 'keras_model' # Special case for Keras
}

ratios = {
    '60:40': 0.4,
    '70:30': 0.3,
    '80:20': 0.2
}

results = []

# --- Keras Model Definition ---
def create_keras_model(input_dim):
    model = Sequential([
        Dense(128, activation='relu', input_dim=input_dim),
        Dropout(0.5),
        Dense(64, activation='relu'),
        Dropout(0.3),
        Dense(1, activation='sigmoid') # Sigmoid for binary classification
    ])
    model.compile(optimizer='adam',
                  loss='binary_crossentropy',
                  metrics=['accuracy'])
    return model

# --- Training Loop ---
for ratio_name, test_size in ratios.items():
    print(f"--- Testing Ratio: {ratio_name} ---")

    # 1. Split data
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=42, stratify=y
    )

    # 2. Scale features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    for model_name, model in models.items():
        print(f"\nTraining {model_name}...")

        if model_name == 'Neural Network':
            # Handle Keras model separately
            nn_model = create_keras_model(X_train_scaled.shape[1])
            nn_model.fit(X_train_scaled, y_train, epochs=20, batch_size=32, verbose=0)
            y_pred_proba = nn_model.predict(X_test_scaled)
            y_pred = (y_pred_proba > 0.5).astype(int)
        else:
            # For sklearn models
            model.fit(X_train_scaled, y_train)
            y_pred = model.predict(X_test_scaled)

        # Evaluate and store results
        accuracy = accuracy_score(y_test, y_pred)
        report = classification_report(y_test, y_pred, target_names=['OK', 'Spam'], output_dict=True)

        results.append({
            'ratio': ratio_name,
            'model': model_name,
            'accuracy': accuracy,
            'spam_precision': report['Spam']['precision'],
            'spam_recall': report['Spam']['recall'],
            'spam_f1_score': report['Spam']['f1-score']
        })

        print(f"--- Results for {model_name} at {ratio_name} ---")
        print(classification_report(y_test, y_pred, target_names=['OK', 'Spam']))
        print("-" * 50)

# Convert results to a DataFrame for easy analysis
results_df = pd.DataFrame(results)

# Sort by the F1-score for the 'Spam' class in descending order
best_results = results_df.sort_values(by='spam_f1_score', ascending=False)

print("--- Model Performance Comparison ---")
print(best_results)

# Identify the best model configuration
best_config = best_results.iloc[0]
print("\n--- Best Performing Model ---")
print(f"Ratio: {best_config['ratio']}")
print(f"Model: {best_config['model']}")
print(f"Accuracy: {best_config['accuracy']:.4f}")
print(f"Spam F1-Score: {best_config['spam_f1_score']:.4f}")

# --- 1. Retrain the best model on ALL data ---
print("Retraining the best model on the entire dataset...")

# In a real scenario, you'd pick best_config['model'] and re-instantiate it.
# For demonstration, let's re-instantiate the best model type found
best_model_type = best_config['model']
if best_model_type == 'Logistic Regression':
    final_model = LogisticRegression(max_iter=1000, random_state=42)
elif best_model_type == 'Gaussian Naive Bayes':
    final_model = GaussianNB()
elif best_model_type == 'Random Forest':
    final_model = RandomForestClassifier(random_state=42)
elif best_model_type == 'Neural Network':
    # Create and compile the Keras model
    final_model = create_keras_model(X.shape[1])
else:
     final_model = RandomForestClassifier(random_state=42) # Default if something goes wrong

final_scaler = StandardScaler()

# Fit the scaler and the model on the full dataset
X_scaled = final_scaler.fit_transform(X)

if best_model_type == 'Neural Network':
     final_model.fit(X_scaled, y, epochs=20, batch_size=32, verbose=0)
else:
    final_model.fit(X_scaled, y)


print("Final model retrained successfully.")

# --- 2. Save the final model and scaler ---
joblib.dump(final_model, 'final_spam_model_spam.joblib')
joblib.dump(final_scaler, 'final_scaler_spam.joblib')
# We also need to save the TF-IDF vectorizer and columns to reconstruct the feature vector
joblib.dump(X.columns.tolist(), 'feature_columns_spam.joblib')



Libraries imported successfully!
spam_features.csv loaded successfully.
Features shape: (675, 428)
Target shape: (675,)
--- Testing Ratio: 60:40 ---

Training Logistic Regression...
--- Results for Logistic Regression at 60:40 ---
              precision    recall  f1-score   support

          OK       0.90      0.94      0.92       146
        Spam       0.92      0.87      0.90       124

    accuracy                           0.91       270
   macro avg       0.91      0.90      0.91       270
weighted avg       0.91      0.91      0.91       270

--------------------------------------------------

Training Gaussian Naive Bayes...
--- Results for Gaussian Naive Bayes at 60:40 ---
              precision    recall  f1-score   support

          OK       0.82      0.75      0.78       146
        Spam       0.73      0.81      0.77       124

    accuracy                           0.77       270
   macro avg       0.77      0.78      0.77       270
weighted avg       0.78      0.77  

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step 
--- Results for Neural Network at 60:40 ---
              precision    recall  f1-score   support

          OK       0.88      0.90      0.89       146
        Spam       0.88      0.85      0.87       124

    accuracy                           0.88       270
   macro avg       0.88      0.88      0.88       270
weighted avg       0.88      0.88      0.88       270

--------------------------------------------------
--- Testing Ratio: 70:30 ---

Training Logistic Regression...
--- Results for Logistic Regression at 70:30 ---
              precision    recall  f1-score   support

          OK       0.88      0.95      0.91       110
        Spam       0.93      0.85      0.89        93

    accuracy                           0.90       203
   macro avg       0.91      0.90      0.90       203
weighted avg       0.90      0.90      0.90       203

--------------------------------------------------

Training Gaussian Naive Bayes...
--- Results for G

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step 
--- Results for Neural Network at 70:30 ---
              precision    recall  f1-score   support

          OK       0.92      0.95      0.94       110
        Spam       0.94      0.90      0.92        93

    accuracy                           0.93       203
   macro avg       0.93      0.93      0.93       203
weighted avg       0.93      0.93      0.93       203

--------------------------------------------------
--- Testing Ratio: 80:20 ---

Training Logistic Regression...
--- Results for Logistic Regression at 80:20 ---
              precision    recall  f1-score   support

          OK       0.92      0.97      0.95        73
        Spam       0.97      0.90      0.93        62

    accuracy                           0.94       135
   macro avg       0.94      0.94      0.94       135
weighted avg       0.94      0.94      0.94       135

--------------------------------------------------

Training Gaussian Naive Bayes...
--- Results for G

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step
--- Results for Neural Network at 80:20 ---
              precision    recall  f1-score   support

          OK       0.94      0.99      0.96        73
        Spam       0.98      0.92      0.95        62

    accuracy                           0.96       135
   macro avg       0.96      0.95      0.95       135
weighted avg       0.96      0.96      0.96       135

--------------------------------------------------
--- Model Performance Comparison ---
    ratio                 model  accuracy  spam_precision  spam_recall  \
11  80:20        Neural Network  0.955556        0.982759     0.919355   
8   80:20   Logistic Regression  0.940741        0.965517     0.903226   
7   70:30        Neural Network  0.931034        0.943820     0.903226   
10  80:20         Random Forest  0.925926        0.906250     0.935484   
2   60:40         Random Forest  0.918519        0.925000     0.895161   
6   70:30         Random Forest  0.916256        0.952381  

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Final model retrained successfully.


['feature_columns_spam.joblib']

In [ ]:
# We need to re-create and fit the TF-IDF vectorizer from the original script
# to process new text consistently.
original_texts_from_df = df['text'].tolist()
# Re-define stopwords
thai_sw = set(thai_stopwords())
english_sw = set(stopwords.words('english'))
custom_sw = {'https', 'http', 'วันที่', 'ขอบคุณ', 'ท่าน', 'www', 'com', 'bit',
    'ค่ะ', 'ครับ', 'คือ', 'ว่า', 'ก็', 'และ', 'หรือ', 'กับ', 'ให้', 'ได้',
    'ไม่', 'มี', 'แล้ว', 'ยัง', 'จะ', 'อย่าง', 'เพียง', 'แต่', 'เมื่อ',
    'จาก', 'ที่', 'ใน', 'ของ', 'โดย', 'บน', 'ซึ่ง', 'ทั้ง', 'อีก', 'จึง',
    'เพราะ', 'เลย', 'ก็ยัง', 'ก็ได้', 'จ้า', 'นะ', 'ล่ะ', 'เดี๋ยว', 'แค่',
    'มาก', 'น้อย', 'เป็น'}
all_stopwords = thai_sw.union(english_sw).union(custom_sw).union(set(string.punctuation))

# Define preprocess function specifically for TF-IDF fitting
def preprocess_for_tfidf(text, stop_words):
    translator = str.maketrans('', '', string.punctuation)
    text = text.translate(translator)
    tokens = word_tokenize(text, engine='newmm', keep_whitespace=False)
    # Ensure tokens with length <= 2 are removed
    cleaned_tokens = [
        word for word in tokens
        if word.lower() not in stop_words and len(word.strip()) > 2 and word and not word.isdigit()
    ]
    return cleaned_tokens


# Create the preprocessed text needed to fit the TF-IDF
# Explicitly call preprocess_for_tfidf here
processed_for_tfidf = [' '.join(preprocess_for_tfidf(text, all_stopwords)) for text in original_texts_from_df]
final_tfidf = TfidfVectorizer(stop_words=list(all_stopwords),
    max_df=0.85,
    min_df=3
    ) # Re-instantiate with the same parameters as the original TF-IDF
final_tfidf.fit(processed_for_tfidf)
joblib.dump(final_tfidf, 'final_tfidf_spam.joblib') # Save it too!

print("Model, scaler, TF-IDF vectorizer, and feature list saved.")


# --- 3. Create the complete prediction function ---

# We need the feature engineering functions from your original script (cell 3)
# Re-defining them here to make this cell self-contained
def preprocess(text, stop_words): # Added stop_words parameter
    translator = str.maketrans('', '', string.punctuation)
    text = text.translate(translator)
    tokens = word_tokenize(text, engine='newmm', keep_whitespace=False)
    # Ensure tokens with length <= 2 are removed
    cleaned_tokens = [
        word for word in tokens
        if word.lower() not in stop_words and len(word.strip()) > 1 and word and not word.isdigit()
    ]
    return cleaned_tokens

# Re-define stopwords, custom_stopwords, custom_spamwords etc. for consistency
thai_sw = set(thai_stopwords())
english_sw = set(stopwords.words('english'))
custom_stopwords = {    'https', 'http', 'วันที่', 'ขอบคุณ', 'ท่าน', 'www', 'com', 'bit',
    'ค่ะ', 'ครับ', 'คือ', 'ว่า', 'ก็', 'และ', 'หรือ', 'กับ', 'ให้', 'ได้',
    'ไม่', 'มี', 'แล้ว', 'ยัง', 'จะ', 'อย่าง', 'เพียง', 'แต่', 'เมื่อ',
    'จาก', 'ที่', 'ใน', 'ของ', 'โดย', 'บน', 'ซึ่ง', 'ทั้ง', 'อีก', 'จึง',
    'เพราะ', 'เลย', 'ก็ยัง', 'ก็ได้', 'จ้า', 'นะ', 'ล่ะ', 'เดี๋ยว', 'แค่',
    'มาก', 'น้อย', 'เป็น'}
custom_spamwords_cell3 = {'ฟรี', 'แถม', 'พิเศษ', '100%', 'ด่วน', 'อันดับ1', 'ถอนไว', 'โบนัส', 'แตกง่าย', 'แจก', 'ลุ้น'}
all_stopwords_prediction = thai_sw.union(english_sw).union(custom_stopwords).union(set(string.punctuation))

promo_words = {'สมัคร', 'ฟรี', 'แถม', 'พิเศษ', 'โบนัส', 'แจก', 'ลด', 'สิทธิ์', 'โปร', 'ดีล', 'คูปอง', 'ส่วนลด', 'สมัครด่วน', 'ยืนยันสิทธิ'}
money_words = {'บาท', 'เงิน', 'จ่าย', 'ค่า', 'บ.', 'บัตรเติมเงิน', 'ชำระ', 'จ่ายเงิน', 'รับเงิน', 'คืนเงิน', 'มูลค่า'}
number_words = set([str(i) for i in range(0, 101)]) | {'เลข', 'เลขเด็ด', 'งวด', 'หวย', 'รางวัล', 'เลขท้าย'}
phone_words = {'โทร', 'ทรู', 'เน็ต', 'เติม', 'ไลน์', 'เบอร์', 'มือถือ', 'โทรศัพท์', 'ติดต่อ', 'line', 'เบอร์โทร'}
urgent_words = {'ด่วน', 'คลิก', 'รีบ', 'อย่า', 'สมัครด่วน', 'ยืนยันสิทธิ', 'อย่ารอช้า', 'เฉพาะคุณ', 'กด', 'เร็ว', 'ทันที'}
luck_words = {'ดวง', 'ดวงดี', 'เฮง', 'โชค', 'โชคดี', 'รวย', 'มงคล', 'หวย', 'รางวัล'}
product_words = {'สินค้า', 'บริการ', 'แพ็ก', 'แพ็กเกจ', 'ดีล', 'ช้อป', 'โปร', 'ส่วนลด', 'คูปอง', 'โปรโมชัน', 'โปรโมชั่น'}

# Redefine feature functions from cell 3 for prediction
def count_urls(text):
    return len(re.findall(r'(http[s]?://|www\.|bit\.ly|cutt\.ly|\.com|\.me|\.co|\.in)', text))

def count_money(text):
    return len(re.findall(r'\d+[ ]*(บาท|฿|บ\.)', text))

def count_digits(text):
    return len(re.findall(r'\d+', text))

def has_line_id(text):
    return int('line' in text.lower() or 'ไลน์' in text.lower())

def has_phone_number(text):
    return int(bool(re.search(r'\b0[689]{1}[0-9]{8}\b', text)))

def spamword_count(tokens):
    return sum(1 for w in tokens if w in custom_spamwords_cell3)

def message_length(text):
    return len(text)

def word_count(tokens):
    return len(tokens)

def digit_ratio(text):
    digits = sum(c.isdigit() for c in text)
    return digits / len(text) if len(text) > 0 else 0

def count_emoji(text):
    return sum(1 for ch in text if emoji.is_emoji(ch))

def count_uppercase_en(text):
    return sum(1 for w in re.findall(r'[A-Z]{2,}', text))

def count_short_link(text):
    return len(re.findall(r'(bit\.ly|tmn\.app\.link|ct\.elinks\.io|m9s\.in|shp\.ee|store\.line\.me)', text))

def count_promo_words(tokens):
    return sum(1 for w in tokens if w in promo_words)

def count_money_words(tokens):
    return sum(1 for w in tokens if w in money_words)

def count_number_words(tokens):
    return sum(1 for w in tokens if w in number_words)

def count_phone_words(tokens):
    return sum(1 for w in tokens if w in phone_words)

def count_urgent_words(tokens):
    return sum(1 for w in tokens if w in urgent_words)

def count_luck_words(tokens):
    return sum(1 for w in tokens if w in luck_words)

def count_product_words(tokens):
    return sum(1 for w in tokens if w in product_words)

def count_ussd_code(text):
    return len(re.findall(r'\*(\d+\*)*\d+#', text))


def create_feature_vector(text, tfidf_vectorizer, feature_columns):
    """Creates a full feature vector from raw text and returns feature values."""

    # Preprocess text using stopwords defined for prediction
    tokens = preprocess(text, all_stopwords_prediction)

    # Calculate ALL engineered features present in cell 3's feature_df
    engineered_features = {
        'url_count': count_urls(text),
        'short_link': count_short_link(text),
        'money_count': count_money(text),
        'digit_count': count_digits(text),
        'digit_ratio': digit_ratio(text),
        'ussd_code': count_ussd_code(text),
        'spamword_count': spamword_count(tokens),
        'has_line_id': has_line_id(text),
        'has_phone_number': has_phone_number(text),
        'emoji_count': count_emoji(text),
        'uppercase_en_count': count_uppercase_en(text),
        'promo_word_count': count_promo_words(tokens),
        'money_word_count': count_money_words(tokens),
        'number_word_count': count_number_words(tokens),
        'phone_word_count': count_phone_words(tokens),
        'urgent_word_count': count_urgent_words(tokens),
        'luck_word_count': count_luck_words(tokens),
        'product_word_count': count_product_words(tokens),
        'msg_length': message_length(text),
        'word_count': word_count(tokens)
    }

    # Convert engineered features to DataFrame
    engineered_features_df = pd.DataFrame([engineered_features])

    # TF-IDF features
    # Ensure consistency in preprocessing for TF-IDF transformation
    processed_text_for_tfidf = ' '.join(preprocess(text, all_stopwords_prediction))
    # Use the passed tfidf_vectorizer (which is the loaded one) to transform
    tfidf_matrix = tfidf_vectorizer.transform([processed_text_for_tfidf]).toarray()
    tfidf_df = pd.DataFrame(tfidf_matrix, columns=tfidf_vectorizer.get_feature_names_out())

    # Combine engineered and TF-IDF features
    combined_df = pd.concat([engineered_features_df, tfidf_df], axis=1)

    # Align columns with the training data (feature_columns)
    # Create a new DataFrame with all expected columns, then fill with values from combined_df
    final_vector = pd.DataFrame(columns=feature_columns)
    final_vector = pd.concat([final_vector, combined_df], ignore_index=True).fillna(0)

    # Explicitly reindex to ensure columns are in the correct order and all training columns are present
    final_vector = final_vector.reindex(columns=feature_columns, fill_value=0)


    # Return both the vector for prediction and the dictionary of actual feature values
    # We filter out TF-IDF features that are zero to avoid cluttering reasons
    active_tfidf_features = {f'TF-IDF: {col}': val for col, val in tfidf_df.iloc[0].items() if val > 0}

    all_feature_values = {**engineered_features, **active_tfidf_features}

    return final_vector[feature_columns], all_feature_values


# --- The Master Prediction Function ---
def predict_message(text):
    """
    Takes a raw text message and predicts if it's a spam, providing reasons and
    percentage contributions of features if it's a spam.
    """
    # Load the saved components
    try:
        # Load the model dynamically based on best_config
        model_filename = 'final_spam_model_spam.joblib' # Using the saved name
        model = joblib.load(model_filename)

        scaler = joblib.load('final_scaler_spam.joblib')
        tfidf = joblib.load('final_tfidf_spam.joblib')
        columns = joblib.load('feature_columns_spam.joblib')
    except FileNotFoundError:
        return "Error: Model components not found. Please run the training cell first."

    # Re-define stopwords and custom spam words (must be consistent with training)
    thai_sw = set(thai_stopwords())
    english_sw = set(stopwords.words('english'))
    custom_stopwords_cell3 = {'https', 'http', 'วันที่', 'ขอบคุณ', 'ท่าน', 'www', '.com'}
    all_stopwords_prediction = thai_sw.union(english_sw).union(custom_stopwords_cell3).union(set(string.punctuation))

    # Redefine the specific spam words used for reasoning if needed, or use the ones for spamword_count
    # For consistency with spamword_count feature, let's use custom_spamwords_cell3
    custom_spamwords_for_reasoning = custom_spamwords_cell3 # Using the set from cell 1/2/3

    # 1. Create feature vector from the new text and get feature values
    # Pass the loaded tfidf object to create_feature_vector
    feature_vector_df, feature_values_dict = create_feature_vector(text, tfidf, columns)

    # 2. Scale the features
    scaled_vector = scaler.transform(feature_vector_df)

    # 3. Make prediction
    if isinstance(model, Sequential): # Handle Keras model prediction
         probability = model.predict(scaled_vector, verbose=0)
         prediction = (probability > 0.5).astype(int)
         probability_spam = probability[0][0] # Keras sigmoid output is single value
         probability_ok = 1 - probability_spam
    else: # Handle sklearn model prediction
        prediction = model.predict(scaled_vector)
        probability = model.predict_proba(scaled_vector)
        probability_ok = probability[0][0]
        probability_spam = probability[0][1]


    reasoning_parts_with_counts = [] # Store reasons as (reason_string, count)

    # Define a threshold for TF-IDF features to be considered "present" for reasoning
    TFIDF_REASONING_THRESHOLD = 0.1

    if prediction[0] == 1: # Predicted as SPAM
        # Add engineered features to reasoning_parts_with_counts if they contribute
        # Prioritize features with non-zero values for reasoning
        if feature_values_dict.get('url_count', 0) > 0:
             reasoning_parts_with_counts.append((f"Contains {feature_values_dict['url_count']} URL(s)", feature_values_dict['url_count']))
        if feature_values_dict.get('short_link', 0) > 0:
             reasoning_parts_with_counts.append((f"Contains {feature_values_dict['short_link']} short link(s)", feature_values_dict['short_link']))
        if feature_values_dict.get('money_count', 0) > 0:
             reasoning_parts_with_counts.append((f"Mentions money/currency {feature_values_dict['money_count']} time(s)", feature_values_dict['money_count']))
        if feature_values_dict.get('digit_count', 0) > 0:
             reasoning_parts_with_counts.append((f"Contains {feature_values_dict['digit_count']} digit sequence(s)", feature_values_dict['digit_count']))
        if feature_values_dict.get('digit_ratio', 0) > 0:
             reasoning_parts_with_counts.append((f"High digit ratio ({feature_values_dict['digit_ratio']:.2f})", feature_values_dict['digit_ratio'] * 10)) # Scale ratio for reasoning weight
        if feature_values_dict.get('ussd_code', 0) > 0:
             reasoning_parts_with_counts.append(("Contains USSD code", feature_values_dict['ussd_code']))
        if feature_values_dict.get('spamword_count', 0) > 0:
             reasoning_parts_with_counts.append((f"Contains {feature_values_dict['spamword_count']} common spam word(s)", feature_values_dict['spamword_count']))
        if feature_values_dict.get('has_line_id', 0) > 0:
             reasoning_parts_with_counts.append(("Mentions LINE ID", feature_values_dict['has_line_id']))
        if feature_values_dict.get('has_phone_number', 0) > 0:
             reasoning_parts_with_counts.append(("Contains a phone number", feature_values_dict['has_phone_number']))
        if feature_values_dict.get('emoji_count', 0) > 0:
             reasoning_parts_with_counts.append((f"Contains {feature_values_dict['emoji_count']} emoji(s)", feature_values_dict['emoji_count']))
        if feature_values_dict.get('uppercase_en_count', 0) > 0:
             reasoning_parts_with_counts.append((f"Contains {feature_values_dict['uppercase_en_count']} uppercase English word(s)", feature_values_dict['uppercase_en_count']))
        if feature_values_dict.get('promo_word_count', 0) > 0:
             reasoning_parts_with_counts.append((f"Contains {feature_values_dict['promo_word_count']} promotional word(s)", feature_values_dict['promo_word_count']))
        if feature_values_dict.get('money_word_count', 0) > 0:
             reasoning_parts_with_counts.append((f"Contains {feature_values_dict['money_word_count']} money-related word(s)", feature_values_dict['money_word_count']))
        if feature_values_dict.get('number_word_count', 0) > 0:
             reasoning_parts_with_counts.append((f"Contains {feature_values_dict['number_word_count']} number-related word(s)", feature_values_dict['number_word_count']))
        if feature_values_dict.get('phone_word_count', 0) > 0:
             reasoning_parts_with_counts.append((f"Contains {feature_values_dict['phone_word_count']} phone-related word(s)", feature_values_dict['phone_word_count']))
        if feature_values_dict.get('urgent_word_count', 0) > 0:
             reasoning_parts_with_counts.append((f"Contains {feature_values_dict['urgent_word_count']} urgent word(s)", feature_values_dict['urgent_word_count']))
        if feature_values_dict.get('luck_word_count', 0) > 0:
             reasoning_parts_with_counts.append((f"Contains {feature_values_dict['luck_word_count']} luck-related word(s)", feature_values_dict['luck_word_count']))
        if feature_values_dict.get('product_word_count', 0) > 0:
             reasoning_parts_with_counts.append((f"Contains {feature_values_dict['product_word_count']} product-related word(s)", feature_values_dict['product_word_count']))
        # Include msg_length and word_count if they are significantly different from average? Or just include always?
        # Let's include them always for now if they are part of the trained features.
        if 'msg_length' in feature_values_dict:
             reasoning_parts_with_counts.append((f"Message length: {feature_values_dict['msg_length']} characters", feature_values_dict['msg_length'] / 10)) # Scale length for reasoning weight
        if 'word_count' in feature_values_dict:
             reasoning_parts_with_counts.append((f"Word count: {feature_values_dict['word_count']}", feature_values_dict['word_count']))


        # Add significant TF-IDF words based on the threshold
        tfidf_features_detected = []
        for feature_name, value in feature_values_dict.items():
            if feature_name.startswith('TF-IDF: ') and value > TFIDF_REASONING_THRESHOLD:
                original_word = feature_name.replace('TF-IDF: ', '')
                # Use the TF-IDF value itself as the weight for sorting/proportion
                tfidf_features_detected.append((f"Keyword '{original_word}' (TF-IDF: {value:.2f})", value))

        # Combine engineered and TF-IDF reasons
        all_reasons_with_weights = reasoning_parts_with_counts + tfidf_features_detected

        # Sort reasons by weight (e.g., count or TF-IDF value) in descending order
        all_reasons_with_weights.sort(key=lambda item: item[1], reverse=True) # Changed ascending=False to reverse=True

        formatted_reasons = []
        if all_reasons_with_weights:
            # Calculate total weight for proportionality
            total_weight = sum(weight for _, weight in all_reasons_with_weights)
            if total_weight > 0:
                for reason_text, weight in all_reasons_with_weights:
                     # Calculate percentage based on weight
                     percentage = (weight / total_weight) * 100
                     formatted_reasons.append(f"{reason_text} ({percentage:.1f}%)")

                reasoning_str = "Reason: " + ", ".join(formatted_reasons)
            else:
                 reasoning_str = "Reason: Model detected spam based on subtle patterns (no distinct features identified for detailed proportion)." # Fallback if all weights are zero

        else:
            reasoning_str = "Reason: Model detected spam based on subtle patterns (no distinct features identified)." # Fallback if no reasons was added


        return f"Prediction: SPAM, {reasoning_str} (Confidence: {probability_spam:.2%})"
    else: # Predicted as OK
        return f"Prediction: OK (Confidence: {probability_ok:.2%})"

# --- Test Cases ---

spam_test_1 = "ด่วน!ค่าโทรหมดกด*158# สมัครดวงดีลุ้นรับค่าโทรฟรี 500บาท 3บ/ขค"
spam_test_2 = "【Thisshop】คูปองเฉพาะหมวดดิจิตอลแจกเข้าบัญชีแล้ว ผ่อนสมาร์ทโฟนที่ Thisshop เริ่มต้นที่ 55.-/วัน https://s.thisshop.com/s/OZPVTVYN"
spam_test_3 = "คลิกเก็บส่วนลด Lazada 7.7 สูงสุด 777.-  ที่ https://ct.elinks.io/L2gSWWhq ดีลลดสูงสุด 90% แถมส่งฟรีไม่มีขั้นต่ำ"
spam_test_4 = "Lazada 7.7ดีลเด็ดลด 90% รับส่วนลดสูงสุด 3500"


ok_test_1 = "กินข้าวเย็นกันไหม"
ok_test_2 = "ลืมหยิบบัตรประชาชนมาด้วย อย่าลืมนะ"
ok_test_3 = "แค่จะมาบอกว่าสู้ๆ นะวันนี้ 😊"
ok_test_4 = "เมื่อไหร่ประชุมเสร็จแล้วบอกด้วยนะ จะไปรับ"


print("--- Testing Spam Messages ---")
print(f"Message: '{spam_test_1}'\n{predict_message(spam_test_1)}\n")
print(f"Message: '{spam_test_2}'\n{predict_message(spam_test_2)}\n")
print(f"Message: '{spam_test_3}'\n{predict_message(spam_test_3)}\n")
print(f"Message: '{spam_test_4}'\n{predict_message(spam_test_4)}\n")


print("--- Testing OK Messages ---")
print(f"Message: '{ok_test_1}'\n{predict_message(ok_test_1)}\n")
print(f"Message: '{ok_test_2}'\n{predict_message(ok_test_2)}\n")
print(f"Message: '{ok_test_3}'\n{predict_message(ok_test_3)}\n")
print(f"Message: '{ok_test_4}'\n{predict_message(ok_test_4)}\n")

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

/usr/local/lib/python3.12/dist-packages/sklearn/feature_extraction/text.py:402: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['กคน', 'กคร', 'กครา', 'กคราว', 'กจะ', 'กช', 'กต', 'กท', 'กทาง', 'กน', 'กระท', 'กระน', 'กระไร', 'กล', 'กว', 'กส', 'กหน', 'กอ', 'กอย', 'กำล', 'กเม', 'กแห', 'กๆ', 'ขณะท', 'ขณะน', 'ขณะหน', 'ขณะเด', 'ขอบค', 'คงอย', 'คร', 'ครบคร', 'ครบถ', 'คราท', 'คราน', 'คราวก', 'คราวท', 'คราวน', 'คราวหน', 'คราวหล', 'คราวโน', 'คราหน', 'คล', 'งก', 'งกระน', 'งกล', 'งกว', 'งข', 'งคง', 'งคน', 'งครา', 'งคราว', 'งง', 'งจ', 'งจน', 'งจะ', 'งจาก', 'งต', 'งท', 'งน', 'งบ', 'งปวง', 'งมวล', 'งละ', 'งว', 'งส', 'งหน', 'งหมด', 'งหมาย', 'งหล', 'งหลาย', 'งอย', 'งเก', 'งเคย', 'งเน', 'งเป', 'งเม', 'งแก', 'งแต', 'งแม', 'งแล', 'งโง', 'งโน', 'งใด', 'งใหญ', 'งไง', 'งได', 'งไหน', 'งๆ', 'งๆจ', 'จก', 'จจ', 'จนกระท', 'จนกว', 'จนขณะน', 'จนถ', 'จนท', 'จนบ', 'จนเม', 'จนแม', 'จร', 'จรดก', 'จวนเจ', 'จวบก', 'จส', 'จสมบ', 'จะได', 'จากน', 'จำเป', '

Model, scaler, TF-IDF vectorizer, and feature list saved.
--- Testing Spam Messages ---
Message: 'ด่วน!ค่าโทรหมดกด*158# สมัครดวงดีลุ้นรับค่าโทรฟรี 500บาท 3บ/ขค'
Prediction: SPAM, Reason: Word count: 12 (29.9%), Message length: 61 characters (15.2%), Contains 3 digit sequence(s) (7.5%), Contains 3 common spam word(s) (7.5%), Contains 3 money-related word(s) (7.5%), Contains 2 promotional word(s) (5.0%), Contains 2 phone-related word(s) (5.0%), Contains 2 urgent word(s) (5.0%), High digit ratio (0.11) (2.9%), Mentions money/currency 1 time(s) (2.5%), Contains USSD code (2.5%), Contains 1 luck-related word(s) (2.5%), Keyword 'กด' (TF-IDF: 0.51) (1.3%), Keyword 'โทร' (TF-IDF: 0.45) (1.1%), Keyword 'วน' (TF-IDF: 0.39) (1.0%), Keyword 'ดวงด' (TF-IDF: 0.36) (0.9%), Keyword 'บาท' (TF-IDF: 0.26) (0.7%), Keyword 'บขค' (TF-IDF: 0.25) (0.6%), Keyword 'ฟร' (TF-IDF: 0.21) (0.5%), Keyword 'สม' (TF-IDF: 0.21) (0.5%), Keyword 'คร' (TF-IDF: 0.21) (0.5%) (Confidence: 100.00%)



/usr/local/lib/python3.12/dist-packages/sklearn/feature_extraction/text.py:402: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['กคน', 'กคร', 'กครา', 'กคราว', 'กจะ', 'กช', 'กต', 'กท', 'กทาง', 'กน', 'กระท', 'กระน', 'กระไร', 'กล', 'กว', 'กส', 'กหน', 'กอ', 'กอย', 'กำล', 'กเม', 'กแห', 'กๆ', 'ขณะท', 'ขณะน', 'ขณะหน', 'ขณะเด', 'ขอบค', 'คงอย', 'คร', 'ครบคร', 'ครบถ', 'คราท', 'คราน', 'คราวก', 'คราวท', 'คราวน', 'คราวหน', 'คราวหล', 'คราวโน', 'คราหน', 'คล', 'งก', 'งกระน', 'งกล', 'งกว', 'งข', 'งคง', 'งคน', 'งครา', 'งคราว', 'งง', 'งจ', 'งจน', 'งจะ', 'งจาก', 'งต', 'งท', 'งน', 'งบ', 'งปวง', 'งมวล', 'งละ', 'งว', 'งส', 'งหน', 'งหมด', 'งหมาย', 'งหล', 'งหลาย', 'งอย', 'งเก', 'งเคย', 'งเน', 'งเป', 'งเม', 'งแก', 'งแต', 'งแม', 'งแล', 'งโง', 'งโน', 'งใด', 'งใหญ', 'งไง', 'งได', 'งไหน', 'งๆ', 'งๆจ', 'จก', 'จจ', 'จนกระท', 'จนกว', 'จนขณะน', 'จนถ', 'จนท', 'จนบ', 'จนเม', 'จนแม', 'จร', 'จรดก', 'จวนเจ', 'จวบก', 'จส', 'จสมบ', 'จะได', 'จากน', 'จำเป', '

Message: '【Thisshop】คูปองเฉพาะหมวดดิจิตอลแจกเข้าบัญชีแล้ว ผ่อนสมาร์ทโฟนที่ Thisshop เริ่มต้นที่ 55.-/วัน https://s.thisshop.com/s/OZPVTVYN'
Prediction: SPAM, Reason: Message length: 128 characters (35.5%), Word count: 12 (33.3%), Contains 2 URL(s) (5.5%), Contains 2 promotional word(s) (5.5%), Contains 1 digit sequence(s) (2.8%), Contains 1 common spam word(s) (2.8%), Contains 1 uppercase English word(s) (2.8%), Contains 1 product-related word(s) (2.8%), Keyword 'สมาร' (TF-IDF: 0.40) (1.1%), Keyword 'าบ' (TF-IDF: 0.40) (1.1%), Keyword 'โฟน' (TF-IDF: 0.40) (1.1%), Keyword 'มต' (TF-IDF: 0.36) (1.0%), Keyword 'ปอง' (TF-IDF: 0.30) (0.8%), Keyword 'ญช' (TF-IDF: 0.29) (0.8%), Keyword 'แจก' (TF-IDF: 0.26) (0.7%), Keyword 'เข' (TF-IDF: 0.25) (0.7%), Keyword 'เร' (TF-IDF: 0.24) (0.7%), Keyword 'อน' (TF-IDF: 0.20) (0.5%), High digit ratio (0.02) (0.4%) (Confidence: 99.93%)



/tmp/ipython-input-4117327438.py:185: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  final_vector = pd.concat([final_vector, combined_df], ignore_index=True).fillna(0)
/tmp/ipython-input-4117327438.py:185: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  final_vector = pd.concat([final_vector, combined_df], ignore_index=True).fillna(0)
/usr/local/lib/python3.12/dist-packages/sklearn/feature_extraction/text.py:402: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens

...

/usr/local/lib/python3.12/dist-packages/sklearn/feature_extraction/text.py:402: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['กคน', 'กคร', 'กครา', 'กคราว', 'กจะ', 'กช', 'กต', 'กท', 'กทาง', 'กน', 'กระท', 'กระน', 'กระไร', 'กล', 'กว', 'กส', 'กหน', 'กอ', 'กอย', 'กำล', 'กเม', 'กแห', 'กๆ', 'ขณะท', 'ขณะน', 'ขณะหน', 'ขณะเด', 'ขอบค', 'คงอย', 'คร', 'ครบคร', 'ครบถ', 'คราท', 'คราน', 'คราวก', 'คราวท', 'คราวน', 'คราวหน', 'คราวหล', 'คราวโน', 'คราหน', 'คล', 'งก', 'งกระน', 'งกล', 'งกว', 'งข', 'งคง', 'งคน', 'งครา', 'งคราว', 'งง', 'งจ', 'งจน', 'งจะ', 'งจาก', 'งต', 'งท', 'งน', 'งบ', 'งปวง', 'งมวล', 'งละ', 'งว', 'งส', 'งหน', 'งหมด', 'งหมาย', 'งหล', 'งหลาย', 'งอย', 'งเก', 'งเคย', 'งเน', 'งเป', 'งเม', 'งแก', 'งแต', 'งแม', 'งแล', 'งโง', 'งโน', 'งใด', 'งใหญ', 'งไง', 'งได', 'งไหน', 'งๆ', 'งๆจ', 'จก', 'จจ', 'จนกระท', 'จนกว', 'จนขณะน', 'จนถ', 'จนท', 'จนบ', 'จนเม', 'จนแม', 'จร', 'จรดก', 'จวนเจ', 'จวบก', 'จส', 'จสมบ', 'จะได', 'จากน', 'จำเป', '

Message: 'ลืมหยิบบัตรประชาชนมาด้วย อย่าลืมนะ'
Prediction: OK (Confidence: 99.91%)



/tmp/ipython-input-4117327438.py:185: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  final_vector = pd.concat([final_vector, combined_df], ignore_index=True).fillna(0)
/tmp/ipython-input-4117327438.py:185: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  final_vector = pd.concat([final_vector, combined_df], ignore_index=True).fillna(0)


Message: 'แค่จะมาบอกว่าสู้ๆ นะวันนี้ 😊'
Prediction: OK (Confidence: 91.62%)

Message: 'เมื่อไหร่ประชุมเสร็จแล้วบอกด้วยนะ จะไปรับ'
Prediction: OK (Confidence: 94.02%)



/usr/local/lib/python3.12/dist-packages/sklearn/feature_extraction/text.py:402: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['กคน', 'กคร', 'กครา', 'กคราว', 'กจะ', 'กช', 'กต', 'กท', 'กทาง', 'กน', 'กระท', 'กระน', 'กระไร', 'กล', 'กว', 'กส', 'กหน', 'กอ', 'กอย', 'กำล', 'กเม', 'กแห', 'กๆ', 'ขณะท', 'ขณะน', 'ขณะหน', 'ขณะเด', 'ขอบค', 'คงอย', 'คร', 'ครบคร', 'ครบถ', 'คราท', 'คราน', 'คราวก', 'คราวท', 'คราวน', 'คราวหน', 'คราวหล', 'คราวโน', 'คราหน', 'คล', 'งก', 'งกระน', 'งกล', 'งกว', 'งข', 'งคง', 'งคน', 'งครา', 'งคราว', 'งง', 'งจ', 'งจน', 'งจะ', 'งจาก', 'งต', 'งท', 'งน', 'งบ', 'งปวง', 'งมวล', 'งละ', 'งว', 'งส', 'งหน', 'งหมด', 'งหมาย', 'งหล', 'งหลาย', 'งอย', 'งเก', 'งเคย', 'งเน', 'งเป', 'งเม', 'งแก', 'งแต', 'งแม', 'งแล', 'งโง', 'งโน', 'งใด', 'งใหญ', 'งไง', 'งได', 'งไหน', 'งๆ', 'งๆจ', 'จก', 'จจ', 'จนกระท', 'จนกว', 'จนขณะน', 'จนถ', 'จนท', 'จนบ', 'จนเม', 'จนแม', 'จร', 'จรดก', 'จวนเจ', 'จวบก', 'จส', 'จสมบ', 'จะได', 'จากน', 'จำเป', '

In [ ]:
import pandas as pd
import numpy as np
import re
import joblib
import string
import nltk
from collections import Counter
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from pythainlp.tokenize import word_tokenize
from pythainlp.corpus.common import thai_stopwords
import emoji # Import emoji

# Ensure NLTK stopwords are available
try:
    nltk.data.find('corpora/stopwords')
except LookupError:
    nltk.download('stopwords')

from nltk.corpus import stopwords

print("Libraries imported successfully!")

# Define stopwords
thai_sw = set(thai_stopwords())
english_sw = set(stopwords.words('english'))
custom_sw = {
    'https', 'http', 'วันที่', 'ขอบคุณ', 'ท่าน', 'www', 'com', 'bit',
    'ค่ะ', 'ครับ', 'คือ', 'ว่า', 'ก็', 'และ', 'หรือ', 'กับ', 'ให้', 'ได้',
    'ไม่', 'มี', 'แล้ว', 'ยัง', 'จะ', 'อย่าง', 'เพียง', 'แต่', 'เมื่อ',
    'จาก', 'ที่', 'ใน', 'ของ', 'โดย', 'บน', 'ซึ่ง', 'ทั้ง', 'อีก', 'จึง',
    'เพราะ', 'เลย', 'ก็ยัง', 'ก็ได้', 'จ้า', 'นะ', 'ล่ะ', 'เดี๋ยว', 'แค่',
    'มาก', 'น้อย', 'เป็น'
}
all_stopwords = thai_sw.union(english_sw).union(custom_sw).union(set(string.punctuation))

# Word categories
custom_spamwords = {'ฟรี', 'แถม', 'พิเศษ', '100%', 'ด่วน', 'อันดับ1', 'ถอนไว', 'โบนัส', 'แตกง่าย', 'แจก', 'ลุ้น'}
promo_words = {'สมัคร', 'ฟรี', 'แถม', 'พิเศษ', 'โบนัส', 'แจก', 'ลด', 'สิทธิ์', 'โปร', 'ดีล', 'คูปอง', 'ส่วนลด', 'สมัครด่วน', 'ยืนยันสิทธิ'}
money_words = {'บาท', 'เงิน', 'จ่าย', 'ค่า', 'บ.', 'บัตรเติมเงิน', 'ชำระ', 'จ่ายเงิน', 'รับเงิน', 'คืนเงิน', 'มูลค่า'}
number_words = set([str(i) for i in range(0, 101)]) | {'เลข', 'เลขเด็ด', 'งวด', 'หวย', 'รางวัล', 'เลขท้าย'}
phone_words = {'โทร', 'ทรู', 'เน็ต', 'เติม', 'ไลน์', 'เบอร์', 'มือถือ', 'โทรศัพท์', 'ติดต่อ', 'line', 'เบอร์โทร'}
urgent_words = {'ด่วน', 'คลิก', 'รีบ', 'อย่า', 'สมัครด่วน', 'ยืนยันสิทธิ', 'อย่ารอช้า', 'เฉพาะคุณ', 'กด', 'เร็ว', 'ทันที'}
luck_words = {'ดวง', 'ดวงดี', 'เฮง', 'โชค', 'โชคดี', 'รวย', 'มงคล', 'หวย', 'รางวัล'}
product_words = {'สินค้า', 'บริการ', 'แพ็ก', 'แพ็กเกจ', 'ดีล', 'ช้อป', 'โปร', 'ส่วนลด', 'คูปอง', 'โปรโมชัน', 'โปรโมชั่น'}

# Preprocess text
def preprocess(text, stop_words):
    text = text.translate(str.maketrans('', '', string.punctuation))
    tokens = word_tokenize(text, engine='newmm', keep_whitespace=False)
    # Removed filtering for token length and digits
    return [w for w in tokens if w.lower() not in stop_words]

# Feature extraction functions
def count_urls(text): return len(re.findall(r'(http[s]?://|www\.|bit\.ly|cutt\.ly|\.com|\.me|\.co|\.in)', text))
def count_money(text): return len(re.findall(r'\d+[ ]*(บาท|บ|฿|บ\.)', text))
def count_digits(text): return len(re.findall(r'\d+', text))
def has_line_id(text): return int('line' in text.lower() or 'ไลน์' in text.lower())
def has_phone_number(text): return int(bool(re.search(r'\b0[689]{1}[0-9]{8}\b', text)))
def spamword_count(tokens): return sum(1 for w in tokens if w in custom_spamwords)
def message_length(text): return len(text)
def word_count(tokens): return len(tokens)
def digit_ratio(text): digits = sum(c.isdigit() for c in text); return digits / len(text) if len(text) > 0 else 0
def count_emoji(text): return sum(1 for ch in text if emoji.is_emoji(ch))
def count_uppercase_en(text): return sum(1 for w in re.findall(r'[A-Z]{2,}', text))
def count_short_link(text): return len(re.findall(r'(bit\.ly|tmn\.app\.link|ct\.elinks\.io|m9s\.in|shp\.ee|store\.line\.me)', text))
def count_promo_words(tokens): return sum(1 for w in tokens if w in promo_words)
def count_money_words(tokens): return sum(1 for w in tokens if w in money_words)
def count_number_words(tokens): return sum(1 for w in tokens if w in number_words)
def count_phone_words(tokens): return sum(1 for w in tokens if w in phone_words)
def count_urgent_words(tokens): return sum(1 for w in tokens if w in urgent_words)
def count_luck_words(tokens): return sum(1 for w in tokens if w in luck_words)
def count_product_words(tokens): return sum(1 for w in tokens if w in product_words)
def count_ussd_code(text): return len(re.findall(r'\*(\d+\*)*\d+#', text))

# Vector creator
def create_feature_vector(text, tfidf_vectorizer, feature_columns):
    tokens = preprocess(text, all_stopwords)
    engineered_features = {
        'url_count': count_urls(text),
        'short_link': count_short_link(text),
        'money_count': count_money(text),
        'digit_count': count_digits(text),
        'digit_ratio': digit_ratio(text),
        'ussd_code': count_ussd_code(text),
        'spamword_count': spamword_count(tokens),
        'has_line_id': has_line_id(text),
        'has_phone_number': has_phone_number(text),
        'emoji_count': count_emoji(text),
        'uppercase_en_count': count_uppercase_en(text),
        'promo_word_count': count_promo_words(tokens),
        'money_word_count': count_money_words(tokens),
        'number_word_count': count_number_words(tokens),
        'phone_word_count': count_phone_words(tokens),
        'urgent_word_count': count_urgent_words(tokens),
        'luck_word_count': count_luck_words(tokens),
        'product_word_count': count_product_words(tokens),
        'msg_length': message_length(text),
        'word_count': word_count(tokens)
    }
    tfidf_matrix = tfidf_vectorizer.transform([' '.join(tokens)]).toarray()
    tfidf_df = pd.DataFrame(tfidf_matrix, columns=tfidf_vectorizer.get_feature_names_out())
    features_df = pd.DataFrame([engineered_features])
    combined_df = pd.concat([features_df, tfidf_df], axis=1)

    # Explicitly reindex to match the training columns
    final_vector = combined_df.reindex(columns=feature_columns, fill_value=0)

    active_tfidf_features = {f'tf_{col}': val for col, val in tfidf_df.iloc[0].items() if val > 0}
    all_feature_values = {**engineered_features, **active_tfidf_features}
    return final_vector, all_feature_values # Return the reindexed DataFrame


# Prediction with explanation
def predict_message(text):
    # Load the model and other necessary files
    try:
        # Load the model dynamically based on best_config
        model_filename = 'final_spam_model_spam.joblib' # Using the saved name
        model = joblib.load(model_filename)

        scaler = joblib.load('final_scaler_spam.joblib')
        tfidf = joblib.load('final_tfidf_spam.joblib')
        columns = joblib.load('feature_columns_spam.joblib')
    except FileNotFoundError:
        return "Error", text, "Model files not found. Please train and save the model first."

    # Use create_feature_vector to get the feature vector and values
    feature_vector_df, feature_values = create_feature_vector(text, tfidf, columns)

    # Debugging prints (Optional - remove after debugging)
    # print(f"Shape of feature_vector_df: {feature_vector_df.shape}")
    # print(f"Number of expected columns (from joblib): {len(columns)}")
    # print(f"Columns in feature_vector_df: {feature_vector_df.columns.tolist()}")
    # print(f"Expected columns (from joblib): {columns}")
    # print(f"Are columns in feature_vector_df same as expected: {feature_vector_df.columns.tolist() == columns}")


    scaled_vector = scaler.transform(feature_vector_df)

    # 3. Make prediction
    if isinstance(model, tf.keras.models.Sequential): # Handle Keras model prediction
         probability = model.predict(scaled_vector, verbose=0)
         prediction = (probability > 0.5).astype(int)
         probability_spam = probability[0][0] # Keras sigmoid output is single value
         probability_ok = 1 - probability_spam
    else: # Handle sklearn model prediction
        prediction = model.predict(scaled_vector)
        probability = model.predict_proba(scaled_vector)
        probability_ok = probability[0][0]
        probability_spam = probability[0][1]


    reasoning_parts_with_counts = [] # Store reasons as (reason_string, count)

    # Define a threshold for TF-IDF features to be considered "present" for reasoning
    TFIDF_REASONING_THRESHOLD = 0.1

    if prediction[0] == 1: # Predicted as SPAM
        if feature_values.get('url_count', 0) > 0:
            reasoning_parts_with_counts.append((f"พบลิงก์ URL จำนวน {feature_values.get('url_count', 0)} ลิงก์", feature_values.get('url_count', 0)))
        if feature_values.get('short_link', 0) > 0:
            reasoning_parts_with_counts.append((f"พบลิงก์แบบย่อจำนวน {feature_values.get('short_link', 0)} ลิงก์", feature_values.get('short_link', 0)))
        if feature_values.get('money_count', 0) > 0:
            reasoning_parts_with_counts.append((f"พบคำเกี่ยวกับเงิน/สกุลเงิน {feature_values.get('money_count', 0)} ครั้ง", feature_values.get('money_count', 0)))
        if feature_values.get('digit_count', 0) > 0:
            reasoning_parts_with_counts.append((f"พบกลุ่มตัวเลข {feature_values.get('digit_count', 0)} กลุ่ม", feature_values.get('digit_count', 0)))
        if feature_values.get('digit_ratio', 0) > 0:
            reasoning_parts_with_counts.append((f"อัตราส่วนตัวเลขสูง ({feature_values.get('digit_ratio', 0):.2f})", feature_values.get('digit_ratio', 0) * 10))
        if feature_values.get('ussd_code', 0) > 0:
            reasoning_parts_with_counts.append((f"พบรหัส USSD เช่น *123#", feature_values.get('ussd_code', 0)))
        if feature_values.get('spamword_count', 0) > 0:
            reasoning_parts_with_counts.append((f"พบคำที่พบบ่อยในข้อความสแปม {feature_values.get('spamword_count', 0)} คำ", feature_values.get('spamword_count', 0)))
        if feature_values.get('has_line_id', 0) > 0:
            reasoning_parts_with_counts.append(("พบการระบุ LINE ID", feature_values.get('has_line_id', 0)))
        if feature_values.get('has_phone_number', 0) > 0:
            reasoning_parts_with_counts.append(("พบหมายเลขโทรศัพท์", feature_values.get('has_phone_number', 0)))
        if feature_values.get('emoji_count', 0) > 0:
            reasoning_parts_with_counts.append((f"พบอีโมจิ {feature_values.get('emoji_count', 0)} ตัว", feature_values.get('emoji_count', 0)))
        if feature_values.get('uppercase_en_count', 0) > 0:
            reasoning_parts_with_counts.append((f"พบคำภาษาอังกฤษตัวพิมพ์ใหญ่ {feature_values.get('uppercase_en_count', 0)} คำ", feature_values.get('uppercase_en_count', 0)))
        if feature_values.get('promo_word_count', 0) > 0:
            reasoning_parts_with_counts.append((f"พบคำเกี่ยวกับโปรโมชั่น {feature_values.get('promo_word_count', 0)} คำ", feature_values.get('promo_word_count', 0)))
        if feature_values.get('money_word_count', 0) > 0:
            reasoning_parts_with_counts.append((f"พบคำที่เกี่ยวกับเงิน {feature_values.get('money_word_count', 0)} คำ", feature_values.get('money_word_count', 0)))
        if feature_values.get('number_word_count', 0) > 0:
            reasoning_parts_with_counts.append((f"พบคำที่เกี่ยวกับตัวเลข {feature_values.get('number_word_count', 0)} คำ", feature_values.get('number_word_count', 0)))
        if feature_values.get('phone_word_count', 0) > 0:
            reasoning_parts_with_counts.append((f"พบคำที่เกี่ยวกับโทรศัพท์ {feature_values.get('phone_word_count', 0)} คำ", feature_values.get('phone_word_count', 0)))
        if feature_values.get('urgent_word_count', 0) > 0:
            reasoning_parts_with_counts.append((f"พบคำเร่งด่วน {feature_values.get('urgent_word_count', 0)} คำ", feature_values.get('urgent_word_count', 0)))
        if feature_values.get('luck_word_count', 0) > 0:
            reasoning_parts_with_counts.append((f"พบคำที่เกี่ยวกับโชค/รางวัล {feature_values.get('luck_word_count', 0)} คำ", feature_values.get('luck_word_count', 0)))
        if feature_values.get('product_word_count', 0) > 0:
            reasoning_parts_with_counts.append((f"พบคำที่เกี่ยวกับสินค้า/บริการ {feature_values.get('product_word_count', 0)} คำ", feature_values.get('product_word_count', 0)))
        if 'msg_length' in feature_values:
            reasoning_parts_with_counts.append((f"ความยาวข้อความ: {feature_values.get('msg_length', 0)} ตัวอักษร", feature_values.get('msg_length', 0) / 10))
        if 'word_count' in feature_values:
            reasoning_parts_with_counts.append((f"จำนวนคำ: {feature_values.get('word_count', 0)} คำ", feature_values.get('word_count', 0)))

        tfidf_features_detected = []
        for feature_name, value in feature_values.items():
            if feature_name.startswith('tf_') and value > TFIDF_REASONING_THRESHOLD:
                original_word = feature_name.replace('tf_', '')
                tfidf_features_detected.append((original_word, value))

        all_reasons_with_weights = reasoning_parts_with_counts + [(f"มีคำสำคัญ TF-IDF: '{', '.join([word for word, _ in tfidf_features_detected])}'", sum([v for _, v in tfidf_features_detected]))] if tfidf_features_detected else reasoning_parts_with_counts

        all_reasons_with_weights.sort(key=lambda item: item[1], reverse=True)

        formatted_reasons = []
        if all_reasons_with_weights:
            total_weight = sum(weight for _, weight in all_reasons_with_weights)
            if total_weight > 0:
                for reason_text, weight in all_reasons_with_weights:
                    percentage = (weight / total_weight) * 100
                    formatted_reasons.append(f"{reason_text} ({percentage:.1f}%)")
                reasoning_str = ", ".join(formatted_reasons)
            else:
                reasoning_str = "เหตุผล: ตรวจพบข้อความสแปมโดยโมเดล แต่ไม่มีคุณลักษณะเด่นที่ชัดเจน"
        else:
            reasoning_str = "เหตุผล: ตรวจพบข้อความสแปมโดยโมเดล แต่ไม่มีคุณลักษณะใดให้แสดงผล"


        return "SPAM", text, reasoning_str
    else: # Predicted as OK
        return "OK", text, "Not a spam."


# Batch prediction
def generate_spam_report(spam_url='https://raw.githubusercontent.com/0-Mini-Peak-1/Thai-SMS-Scam-Spam-OTP-Dataset/refs/heads/main/spam.txt',
                         output_file_path='spam_analysis_report.csv'):
    report_data = []

    try:
        response = requests.get(spam_url)
        response.encoding = 'utf-8'
        if response.status_code == 200:
            raw_messages = response.text.strip().splitlines()
        else:
            print(f"Failed to fetch data from URL: {spam_url}")
            return
    except Exception as e:
        print("Error loading data from URL:", e)
        return

    print("Generating report...")
    for line in raw_messages:
        line = line.strip()
        if not line:
            continue
        original_message = line[5:].strip() if line.lower().startswith(('spam|', 'ok|')) else line
        # Call predict_message with only the message
        label, data, reason = predict_message(original_message)
        report_data.append({'Label': label, 'Data': data, 'Reason': reason})

    if report_data:
        pd.DataFrame(report_data).to_csv(output_file_path, index=False, encoding='utf-8-sig')
        print(f"Report saved to '{output_file_path}'.")
    else:
        print("No spam messages detected.")
  # Create dummy file and run
if __name__ == "__main__":
    try:
        with open('spam.txt', 'x', encoding='utf-8') as f:
            f.write("spam|ด่วน!ค่าโทรหมดกด*158# สมัครดวงดีลุ้นรับค่าโทรฟรี 500บาท 3บ/ขค\n")
            f.write("spam|มีบัตรเติมเงิน 500บ.มาแจก ลุ้นง่ายๆแค่กด *172# สมัครดวงดี3บ/ขค\n")
            f.write("spam|ซื้อพิซซ่าลุ้นทองทุกสัปดาห์ พร้อมโปร1แถม1! สั่งเลย กด app.1112.com/pno\n")
            f.write("ok|ประชุมพรุ่งนี้ 10 โมงนะ\n")
            f.write("spam|Shopee โปรกลางเดือน โค้ดลด 50% + โค้ดส่งฟรี คลิก m9s.in/r61n66\n")
            f.write("spam|คุ้มสุด 8.8 วันนี้! ลดสูงสุด 88% +เก็บโค้ดลด888บ.ช้อปเลย m9s.in/h36y43\n")
        print("Created 'spam.txt' for testing.")
    except FileExistsError:
        print("'spam.txt' already exists. Using existing file.")

    generate_spam_report('https://raw.githubusercontent.com/0-Mini-Peak-1/Thai-SMS-Scam-Spam-OTP-Dataset/refs/heads/main/spam.txt', 'spam_analysis_report.csv')

Libraries imported successfully!
Created 'spam.txt' for testing.
Generating report...


/usr/local/lib/python3.12/dist-packages/sklearn/feature_extraction/text.py:402: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['กคน', 'กคร', 'กครา', 'กคราว', 'กจะ', 'กช', 'กต', 'กท', 'กทาง', 'กน', 'กระท', 'กระน', 'กระไร', 'กล', 'กว', 'กส', 'กหน', 'กอ', 'กอย', 'กำล', 'กเม', 'กแห', 'กๆ', 'ขณะท', 'ขณะน', 'ขณะหน', 'ขณะเด', 'ขอบค', 'คงอย', 'คร', 'ครบคร', 'ครบถ', 'คราท', 'คราน', 'คราวก', 'คราวท', 'คราวน', 'คราวหน', 'คราวหล', 'คราวโน', 'คราหน', 'คล', 'งก', 'งกระน', 'งกล', 'งกว', 'งข', 'งคง', 'งคน', 'งครา', 'งคราว', 'งง', 'งจ', 'งจน', 'งจะ', 'งจาก', 'งต', 'งท', 'งน', 'งบ', 'งปวง', 'งมวล', 'งละ', 'งว', 'งส', 'งหน', 'งหมด', 'งหมาย', 'งหล', 'งหลาย', 'งอย', 'งเก', 'งเคย', 'งเน', 'งเป', 'งเม', 'งแก', 'งแต', 'งแม', 'งแล', 'งโง', 'งโน', 'งใด', 'งใหญ', 'งไง', 'งได', 'งไหน', 'งๆ', 'งๆจ', 'จก', 'จจ', 'จนกระท', 'จนกว', 'จนขณะน', 'จนถ', 'จนท', 'จนบ', 'จนเม', 'จนแม', 'จร', 'จรดก', 'จวนเจ', 'จวบก', 'จส', 'จสมบ', 'จะได', 'จากน', 'จำเป', '

Report saved to 'spam_analysis_report.csv'.
